# Random-Timing Benchmark

### Setup + Config

In [ ]:
from __future__ import annotations

import logging
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('random_timing_nb')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise FileNotFoundError(f'Cannot locate project root from cwd={Path.cwd()}')

SCRIPT_PATH = PROJECT_ROOT / 'run_random_timing.py'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'random_timing-test'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

K = 1000
LAMBDAS = [0.5, 1.0, 2.0]
SEED = 42
CHUNK_SIZE = 200

INPUTS = [
    PROJECT_ROOT / 'results' / 'poses_true' / 'positions_true_oos_long.parquet',
    PROJECT_ROOT / 'results' / 'runner_exp' / '1d' / 'returns_oos.parquet',
    PROJECT_ROOT / 'results' / 'runner_exp' / '4h' / 'returns_oos.parquet',
    PROJECT_ROOT / 'results' / 'runner_onchain' / 'returns_oos.parquet',
    PROJECT_ROOT / 'results' / 'runner_exp' / '1d' / 'bench_returns_oos.parquet',
    PROJECT_ROOT / 'results' / 'runner_exp' / '4h' / 'bench_returns_oos.parquet',
]
missing = [str(p) for p in INPUTS if not p.exists()]
if missing:
    raise FileNotFoundError(f'Missing required inputs: {missing}')

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SCRIPT_PATH  =', SCRIPT_PATH)
print('RESULTS_DIR  =', RESULTS_DIR)
print('K/LAMBDAS/SEED/CHUNK =', K, LAMBDAS, SEED, CHUNK_SIZE)


PROJECT_ROOT = c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
SCRIPT_PATH  = c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\run_random_timing.py
RESULTS_DIR  = c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\random_timing-test
K/LAMBDAS/SEED/CHUNK = 1000 [0.5, 1.0, 2.0] 42 200


### Запуск скрипта

In [ ]:
import subprocess
import sys
import time

cmd = [
    sys.executable,
    '-u',
    str(SCRIPT_PATH),
    '--k', str(K),
    '--lambdas', *[str(x) for x in LAMBDAS],
    '--seed', str(SEED),
    '--chunk-size', str(CHUNK_SIZE),
    '--log-level', 'INFO',
]

print('Running:', ' '.join(cmd))
t0 = time.perf_counter()

proc = subprocess.Popen(
    cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    encoding='utf-8',
    errors='replace',
    bufsize=1,
)

assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='')

ret = proc.wait()
dt = time.perf_counter() - t0
print(f'Finished in {dt:.1f}s with return code={ret}')
if ret != 0:
    raise RuntimeError(f'run_random_timing.py failed with code {ret}')


Running: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\.venv\Scripts\python.exe -u c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\run_random_timing.py --k 1000 --lambdas 0.5 1.0 2.0 --seed 42 --chunk-size 200 --log-level INFO
2026-05-10 22:17:41,242 | INFO | PROJECT_ROOT=c:\Users\prosh\OneDrive\Рабочий стол\git\diploma
2026-05-10 22:17:41,242 | INFO | OUT_DIR=c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\random_timing-test
2026-05-10 22:17:41,242 | INFO | Params: k=1000 lambdas=[0.5, 1.0, 2.0] seed=42 chunk_size=200
2026-05-10 22:17:44,944 | INFO | positions shape=(2301366, 7)
2026-05-10 22:17:44,944 | INFO | returns_all shape=(2301366, 7)
2026-05-10 22:17:44,944 | INFO | bench_all shape=(209400, 7)
2026-05-10 22:18:12,417 | INFO | Progress: 100/1260 tests done (skipped=0)
2026-05-10 22:18:36,250 | INFO | Progress: 200/1260 tests done (skipped=0)
2026-05-10 22:18:51,829 | INFO | Progress: 300/1260 tests done (skipped=0)
2026-05-10 22:19:12,770 | INFO | Progress: 400/126

In [ ]:
import json

results_path = RESULTS_DIR / 'random_timing_results.parquet'
summary_path = RESULTS_DIR / 'random_timing_summary.parquet'
run_info_path = RESULTS_DIR / 'random_timing_run_info.json'

if not results_path.exists():
    raise FileNotFoundError(results_path)
if not summary_path.exists():
    raise FileNotFoundError(summary_path)
if not run_info_path.exists():
    raise FileNotFoundError(run_info_path)

rt = pd.read_parquet(results_path)
sm = pd.read_parquet(summary_path)
run_info = json.loads(run_info_path.read_text(encoding='utf-8'))

print('results shape:', rt.shape)
print('summary shape:', sm.shape)
print('cost_model:', run_info.get('params', {}).get('cost_model'))
print('entry_bps_by_cost:', run_info.get('params', {}).get('entry_bps_by_cost'))
print('exit_bps_by_cost:', run_info.get('params', {}).get('exit_bps_by_cost'))
print('roundtrip_bps_by_cost:', run_info.get('params', {}).get('roundtrip_bps_by_cost'))

coverage = (
    rt.groupby(['timeframe', 'scenario'], as_index=False)
    .agg(
        rows=('strategy_id', 'size'),
        symbols=('symbol', 'nunique'),
        costs=('cost', 'nunique'),
        strategies=('strategy_id', 'nunique'),
        skipped=('skip_reason', lambda x: int(x.notna().sum())),
    )
    .sort_values(['timeframe', 'scenario'])
    .reset_index(drop=True)
)

sig_share = (
    rt[rt['skip_reason'].isna()]
    .groupby(['timeframe', 'scenario', 'family', 'lam'], as_index=False)['timing_significant']
    .mean()
    .rename(columns={'timing_significant': 'significant_share'})
    .sort_values(['timeframe', 'scenario', 'lam', 'family'])
    .reset_index(drop=True)
)

top_p = rt[rt['p_value'].notna()].sort_values('p_value', ascending=True).head(20).reset_index(drop=True)
bot_p = rt[rt['p_value'].notna()].sort_values('p_value', ascending=False).head(20).reset_index(drop=True)

display(coverage)
display(sig_share)
display(top_p[['timeframe','scenario','symbol','cost','strategy_id','lam','p_value','excess_vs_BH']])
display(bot_p[['timeframe','scenario','symbol','cost','strategy_id','lam','p_value','excess_vs_BH']])


results shape: (1260, 20)
summary shape: (30, 9)
cost_model: entry=commission+spread, exit=commission
entry_bps_by_cost: {'Low': 8.5, 'Base': 12.000000000000002, 'High': 20.0}
exit_bps_by_cost: {'Low': 7.5, 'Base': 10.0, 'High': 15.0}
roundtrip_bps_by_cost: {'Low': 16.0, 'Base': 22.0, 'High': 35.0}


,timeframe,scenario,rows,symbols,costs,strategies,skipped
0,1d,u12,540,5,3,12,0
1,1d,u17_onchain,180,4,3,5,0
2,4h,u12,540,5,3,12,0


,timeframe,scenario,family,lam,significant_share
0,1d,u12,MeanRev,0.5,0.000000
1,1d,u12,Momentum,0.5,0.000000
2,1d,u12,Synergy,0.5,0.040000
3,1d,u12,Trend,0.5,0.000000
4,1d,u12,MeanRev,1.0,0.000000
5,1d,u12,Momentum,1.0,0.000000
6,1d,u12,Synergy,1.0,0.093333
7,1d,u12,Trend,1.0,0.000000
8,1d,u12,MeanRev,2.0,0.000000
9,1d,u12,Momentum,2.0,0.000000


,timeframe,scenario,symbol,cost,strategy_id,lam,p_value,excess_vs_BH
0,1d,u12,XRPUSDT,Base,S5:Ensemble_3Signals,2.0,0.000999,0.010257
1,1d,u12,XRPUSDT,Low,S5:Ensemble_3Signals,2.0,0.000999,0.010248
2,1d,u12,XRPUSDT,High,S5:Ensemble_3Signals,2.0,0.000999,0.010274
3,1d,u12,XRPUSDT,High,S5:Ensemble_3Signals,1.0,0.002997,0.004166
4,1d,u12,XRPUSDT,Low,S5:Ensemble_3Signals,1.0,0.004995,0.004172
5,4h,u12,XRPUSDT,Base,R2:Bollinger_MR,0.5,0.006993,0.000873
6,4h,u12,XRPUSDT,Low,R2:Bollinger_MR,0.5,0.007992,0.000871
7,1d,u12,XRPUSDT,Base,S5:Ensemble_3Signals,1.0,0.007992,0.004170
8,4h,u12,XRPUSDT,Base,R2:Bollinger_MR,1.0,0.012987,0.001985
9,4h,u12,XRPUSDT,High,R2:Bollinger_MR,0.5,0.013986,0.000854


,timeframe,scenario,symbol,cost,strategy_id,lam,p_value,excess_vs_BH
0,1d,u12,BTCUSDT,High,R1:RSI_MR,1.0,0.982018,0.003264
1,4h,u12,XRPUSDT,Low,S4:MA_Confirm_TSMOM,0.5,0.977023,0.000456
2,4h,u12,BNBUSDT,Base,R1:RSI_MR,2.0,0.974026,0.004474
3,4h,u12,BNBUSDT,Low,R1:RSI_MR,1.0,0.974026,0.002056
4,1d,u12,BTCUSDT,Base,R3:ZScore_MR,0.5,0.970030,0.000877
5,1d,u12,BTCUSDT,High,R2:Bollinger_MR,0.5,0.969031,0.000698
6,4h,u12,BNBUSDT,Low,R1:RSI_MR,2.0,0.969031,0.004460
7,4h,u12,XRPUSDT,Base,S4:MA_Confirm_TSMOM,0.5,0.969031,0.000467
8,1d,u12,BTCUSDT,Base,R1:RSI_MR,1.0,0.968032,0.003286
9,1d,u12,BTCUSDT,Base,R2:Bollinger_MR,0.5,0.965035,0.000704


### Интерпретация

In [ ]:
x = rt[rt['skip_reason'].isna()].copy()
if x.empty:
    display(pd.DataFrame(columns=['timeframe','scenario','lam','n_tests','pct_significant_all','pct_significant_trend_synergy','pct_significant_meanrev','scenario_label']))
else:
    x['is_trend_synergy'] = x['family'].isin(['Trend', 'Synergy', 'Synergy+OC'])
    x['is_meanrev'] = x['family'].isin(['MeanRev', 'MeanRev+OC'])

    rows = []
    for (tf, sc, lam), g in x.groupby(['timeframe', 'scenario', 'lam']):
        p_all = float(g['timing_significant'].mean())
        p_ts = float(g.loc[g['is_trend_synergy'], 'timing_significant'].mean()) if g['is_trend_synergy'].any() else float('nan')
        p_mr = float(g.loc[g['is_meanrev'], 'timing_significant'].mean()) if g['is_meanrev'].any() else float('nan')

        if p_all >= 0.60:
            label = 'A (timing broadly significant)'
        elif (pd.notna(p_ts) and p_ts >= 0.60) and (pd.notna(p_mr) and p_mr < 0.40):
            label = 'B (trend/synergy timing dominates)'
        else:
            label = 'C (timing not broadly significant)'

        rows.append({
            'timeframe': tf,
            'scenario': sc,
            'lam': float(lam),
            'n_tests': int(len(g)),
            'pct_significant_all': p_all,
            'pct_significant_trend_synergy': p_ts,
            'pct_significant_meanrev': p_mr,
            'scenario_label': label,
        })

    interp = pd.DataFrame(rows).sort_values(['timeframe', 'scenario', 'lam']).reset_index(drop=True)
    display(interp)


,timeframe,scenario,lam,n_tests,pct_significant_all,pct_significant_trend_synergy,pct_significant_meanrev,scenario_label
0,1d,u12,0.5,180,0.016667,0.025000,0.000000,C (timing not broadly significant)
1,1d,u12,1.0,180,0.038889,0.058333,0.000000,C (timing not broadly significant)
2,1d,u12,2.0,180,0.016667,0.025000,0.000000,C (timing not broadly significant)
3,1d,u17_onchain,0.5,60,0.100000,0.125000,0.083333,C (timing not broadly significant)
4,1d,u17_onchain,1.0,60,0.100000,0.125000,0.083333,C (timing not broadly significant)
5,1d,u17_onchain,2.0,60,0.000000,0.000000,0.000000,C (timing not broadly significant)
6,4h,u12,0.5,180,0.033333,0.025000,0.066667,C (timing not broadly significant)
7,4h,u12,1.0,180,0.038889,0.025000,0.088889,C (timing not broadly significant)
8,4h,u12,2.0,180,0.038889,0.025000,0.088889,C (timing not broadly significant)


##  Значимость по таймингу победителей SPA (не актуально)

In [ ]:
from pathlib import Path
import importlib.util
import pandas as pd

from src.rc_spa import build_universe_matrix, spa_better_models_by_utility

if 'rt' not in globals():
    rt = pd.read_parquet(RESULTS_DIR / 'random_timing_results.parquet')

if not bool(importlib.util.find_spec('arch')):
    raise RuntimeError('arch is required to compute full SPA winners for all lambdas. Install arch in this kernel.')

rt_ok = rt[rt['skip_reason'].isna()].copy()
rt_ok['scenario_spa'] = rt_ok['scenario'].replace({'u17_onchain': 'u17'}).astype(str)

spa_cfg = {
    'reps': 2000,
    'seed': 42,
    'studentize': True,
    'alpha': 0.05,
    'pvalue_type': 'consistent',
}
block_size_by_tf = {'1d': 10, '4h': 60}

winners_cache = RESULTS_DIR / 'spa_winners_full_from_rc_spa.parquet'
force_rebuild = False

def ensure_date_col(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    if 'date' not in df.columns:
        if 'datetime_utc' in df.columns:
            df = df.rename(columns={'datetime_utc': 'date'})
        elif 'index' in df.columns:
            df = df.rename(columns={'index': 'date'})
        else:
            raise KeyError(f'No date-like column. Columns: {list(df.columns)}')
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    return df.dropna(subset=['date'])

if winners_cache.exists() and not force_rebuild:
    winners_full = pd.read_parquet(winners_cache).copy()
    print('Loaded winners cache:', winners_cache)
else:
    base_1d_returns = ensure_date_col(pd.read_parquet(PROJECT_ROOT / 'results' / 'runner_exp' / '1d' / 'returns_oos.parquet'))
    base_1d_bench = ensure_date_col(pd.read_parquet(PROJECT_ROOT / 'results' / 'runner_exp' / '1d' / 'bench_returns_oos.parquet'))
    onchain_1d_returns = ensure_date_col(pd.read_parquet(PROJECT_ROOT / 'results' / 'runner_onchain' / 'returns_oos.parquet'))
    base_4h_returns = ensure_date_col(pd.read_parquet(PROJECT_ROOT / 'results' / 'runner_exp' / '4h' / 'returns_oos.parquet'))
    base_4h_bench = ensure_date_col(pd.read_parquet(PROJECT_ROOT / 'results' / 'runner_exp' / '4h' / 'bench_returns_oos.parquet'))

    returns_u12_1d = base_1d_returns.copy()
    bench_u12_1d = base_1d_bench.copy()
    returns_u17_1d = pd.concat([base_1d_returns, onchain_1d_returns], ignore_index=True)
    bench_u17_1d = base_1d_bench.copy()
    returns_u12_4h = base_4h_returns.copy()
    bench_u12_4h = base_4h_bench.copy()

    universe_map = {
        ('1d', 'u12'): (returns_u12_1d, bench_u12_1d),
        ('1d', 'u17'): (returns_u17_1d, bench_u17_1d),
        ('4h', 'u12'): (returns_u12_4h, bench_u12_4h),
    }

    combos = (
        rt_ok[['timeframe', 'scenario_spa', 'symbol', 'cost', 'lam']]
        .drop_duplicates()
        .sort_values(['timeframe', 'scenario_spa', 'symbol', 'cost', 'lam'])
        .reset_index(drop=True)
    )

    rows = []
    for i, r in combos.iterrows():
        tf = str(r['timeframe'])
        sc = str(r['scenario_spa'])
        sym = str(r['symbol'])
        cost = str(r['cost'])
        lam = float(r['lam'])

        key = (tf, sc)
        if key not in universe_map:
            continue

        ret_long, bench_long = universe_map[key]
        returns_alt, returns_bench = build_universe_matrix(
            ret_long,
            bench_long,
            symbol=sym,
            cost=cost,
        )

        winners = spa_better_models_by_utility(
            returns_alt,
            returns_bench,
            lam=lam,
            block_size=int(block_size_by_tf[tf]),
            reps=int(spa_cfg['reps']),
            seed=int(spa_cfg['seed']),
            studentize=bool(spa_cfg['studentize']),
            alpha=float(spa_cfg['alpha']),
            pvalue_type=str(spa_cfg['pvalue_type']),
        )

        for sid in winners:
            rows.append(
                {
                    'timeframe': tf,
                    'scenario_spa': sc,
                    'symbol': sym,
                    'cost': cost,
                    'lam': lam,
                    'strategy_id': str(sid),
                }
            )

        if (i + 1) % 10 == 0 or (i + 1) == len(combos):
            print(f'SPA winners progress: {i + 1}/{len(combos)} combos processed')

    winners_full = pd.DataFrame(
        rows,
        columns=['timeframe', 'scenario_spa', 'symbol', 'cost', 'lam', 'strategy_id'],
    )
    winners_full = winners_full.drop_duplicates().reset_index(drop=True)
    winners_full.to_parquet(winners_cache, index=False)
    print('Saved winners cache:', winners_cache)

if winners_full.empty:
    raise RuntimeError('SPA full winners set is empty. Cannot compute significance on winners.')

rt_spa = rt_ok.merge(
    winners_full,
    on=['timeframe', 'scenario_spa', 'symbol', 'cost', 'lam', 'strategy_id'],
    how='inner',
)

print('rt_ok rows:', len(rt_ok))
print('winners_full rows:', len(winners_full))
print('rt_spa matched rows:', len(rt_spa))

pvalues_detail_path = RESULTS_DIR / 'random_timing_spa_winners_pvalues_detailed.parquet'
summary_path_spa = RESULTS_DIR / 'random_timing_spa_winners_significance_summary.parquet'
coverage_path_spa = RESULTS_DIR / 'random_timing_spa_winners_coverage_summary.parquet'


if rt_spa.empty:
    rt_spa_empty = pd.DataFrame(columns=['timeframe', 'scenario', 'symbol', 'cost', 'lam', 'strategy_id', 'timing_significant', 'p_value'])
    spa_sig_empty = pd.DataFrame(columns=['symbol', 'timeframe', 'cost', 'lam', 'n_winners', 'n_tests', 'n_significant', 'pct_significant'])
    cov_empty = pd.DataFrame(columns=['timeframe', 'cost', 'lam', 'combos_total', 'combos_with_winners', 'combos_with_match'])
    rt_spa_empty.to_parquet(pvalues_detail_path, index=False)
    spa_sig_empty.to_parquet(summary_path_spa, index=False)
    cov_empty.to_parquet(coverage_path_spa, index=False)
    print('Saved empty SPA outputs:')
    print(' -', pvalues_detail_path)
    print(' -', summary_path_spa)
    print(' -', coverage_path_spa)
    print('No overlap between random_timing results and SPA winners.')
else:
    spa_sig_by_combo = (
        rt_spa.groupby(['symbol', 'timeframe', 'cost', 'lam'], as_index=False)
        .agg(
            n_winners=('strategy_id', 'nunique'),
            n_tests=('timing_significant', 'size'),
            n_significant=('timing_significant', 'sum'),
        )
    )
    spa_sig_by_combo['n_significant'] = spa_sig_by_combo['n_significant'].astype(int)
    spa_sig_by_combo['n_tests'] = spa_sig_by_combo['n_tests'].astype(int)
    spa_sig_by_combo['pct_significant'] = spa_sig_by_combo['n_significant'] / spa_sig_by_combo['n_tests']
    spa_sig_by_combo = spa_sig_by_combo.sort_values(['timeframe', 'symbol', 'cost', 'lam']).reset_index(drop=True)

    combos_rt = rt_ok[['symbol', 'timeframe', 'cost', 'lam']].drop_duplicates()
    combos_w = winners_full[['symbol', 'timeframe', 'cost', 'lam']].drop_duplicates()
    combos_m = rt_spa[['symbol', 'timeframe', 'cost', 'lam']].drop_duplicates()

    cov = combos_rt.merge(
        combos_w.assign(has_winners=1),
        on=['symbol', 'timeframe', 'cost', 'lam'],
        how='left',
    )
    cov['has_winners'] = cov['has_winners'].fillna(0).astype(int)
    cov = cov.merge(
        combos_m.assign(has_match=1),
        on=['symbol', 'timeframe', 'cost', 'lam'],
        how='left',
    )
    cov['has_match'] = cov['has_match'].fillna(0).astype(int)
    cov_summary = (
        cov.groupby(['timeframe', 'cost', 'lam'], as_index=False)
        .agg(
            combos_total=('symbol', 'size'),
            combos_with_winners=('has_winners', 'sum'),
            combos_with_match=('has_match', 'sum'),
        )
        .sort_values(['timeframe', 'cost', 'lam'])
        .reset_index(drop=True)
    )

    rt_spa_pvalues = rt_spa[
        ['timeframe', 'scenario', 'symbol', 'cost', 'lam', 'strategy_id', 'timing_significant', 'p_value']
    ].sort_values(['timeframe', 'scenario', 'symbol', 'cost', 'lam', 'p_value']).reset_index(drop=True)
    rt_spa_pvalues.to_parquet(pvalues_detail_path, index=False)
    spa_sig_by_combo.to_parquet(summary_path_spa, index=False)
    cov_summary.to_parquet(coverage_path_spa, index=False)
    print('Saved SPA outputs:')
    print(' -', pvalues_detail_path)
    print(' -', summary_path_spa)
    print(' -', coverage_path_spa)

    print('\nSPA-confirmed significance by (symbol, timeframe, cost, lam):')
    display(spa_sig_by_combo)

    print('\nCoverage summary by (timeframe, cost, lam):')
    display(cov_summary)

    print('\nMatched SPA winners sample:')
    display(
        rt_spa[
            ['timeframe', 'scenario', 'symbol', 'cost', 'lam', 'strategy_id', 'timing_significant', 'p_value']
        ]
        .sort_values(['timeframe', 'scenario', 'symbol', 'cost', 'lam', 'p_value'])
        .head(80)
        .reset_index(drop=True)
    )


SPA winners progress: 10/126 combos processed
SPA winners progress: 20/126 combos processed
SPA winners progress: 30/126 combos processed
SPA winners progress: 40/126 combos processed
SPA winners progress: 50/126 combos processed
SPA winners progress: 60/126 combos processed
SPA winners progress: 70/126 combos processed
SPA winners progress: 80/126 combos processed
SPA winners progress: 90/126 combos processed
SPA winners progress: 100/126 combos processed
SPA winners progress: 110/126 combos processed
SPA winners progress: 120/126 combos processed
SPA winners progress: 126/126 combos processed
Saved winners cache: c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\random_timing-test\spa_winners_full_from_rc_spa.parquet
rt_ok rows: 1260
winners_full rows: 1313
rt_spa matched rows: 1034
Saved SPA outputs:
 - c:\Users\prosh\OneDrive\Рабочий стол\git\diploma\results\random_timing-test\random_timing_spa_winners_pvalues_detailed.parquet
 - c:\Users\prosh\OneDrive\Рабочий стол\git\dipl

,symbol,timeframe,cost,lam,n_winners,n_tests,n_significant,pct_significant
0,BNBUSDT,1d,Base,1.0,9,9,1,0.111111
1,BNBUSDT,1d,Base,2.0,12,12,0,0.000000
2,BNBUSDT,1d,High,1.0,9,9,0,0.000000
3,BNBUSDT,1d,High,2.0,12,12,0,0.000000
4,BNBUSDT,1d,Low,1.0,9,9,0,0.000000
...,...,...,...,...,...,...,...,...
82,XRPUSDT,4h,High,1.0,12,12,2,0.166667
83,XRPUSDT,4h,High,2.0,12,12,1,0.083333
84,XRPUSDT,4h,Low,0.5,10,10,2,0.200000
85,XRPUSDT,4h,Low,1.0,12,12,2,0.166667



Coverage summary by (timeframe, cost, lam):


,timeframe,cost,lam,combos_total,combos_with_winners,combos_with_match
0,1d,Base,0.5,5,4,4
1,1d,Base,1.0,5,5,5
2,1d,Base,2.0,5,5,5
3,1d,High,0.5,5,4,4
4,1d,High,1.0,5,5,5
5,1d,High,2.0,5,5,5
6,1d,Low,0.5,5,4,4
7,1d,Low,1.0,5,5,5
8,1d,Low,2.0,5,5,5
9,4h,Base,0.5,5,5,5



Matched SPA winners sample:


,timeframe,scenario,symbol,cost,lam,strategy_id,timing_significant,p_value
0,1d,u12,BNBUSDT,Base,1.0,S5:Ensemble_3Signals,True,0.046953
1,1d,u12,BNBUSDT,Base,1.0,S4:MA_Confirm_TSMOM,False,0.160839
2,1d,u12,BNBUSDT,Base,1.0,S3:Breakout_Confirm_MA,False,0.210789
3,1d,u12,BNBUSDT,Base,1.0,T3:Donchian_Breakout,False,0.289710
4,1d,u12,BNBUSDT,Base,1.0,S2:MA200Filter_Bollinger_MR,False,0.453546
...,...,...,...,...,...,...,...,...
75,1d,u12,BTCUSDT,Base,1.0,R1:RSI_MR,False,0.968032
76,1d,u12,BTCUSDT,Base,2.0,T3:Donchian_Breakout,False,0.335664
77,1d,u12,BTCUSDT,Base,2.0,S5:Ensemble_3Signals,False,0.411588
78,1d,u12,BTCUSDT,Base,2.0,S3:Breakout_Confirm_MA,False,0.450549
